> 📄 English: [langgraph_tutorial.ipynb](langgraph_tutorial.ipynb)
![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--langgraph-agent--langgraph-tutorial)

# LangGraph 教程：构建文本分析流水线

## LangGraph 简介

LangGraph 是 LangChain 推出的一个强大框架，用于借助 LLM 构建有状态、多参与者的应用。它通过基于图的方式，提供了构建复杂 AI Agent 所需的结构与工具。

可以把 LangGraph 想象成建筑师的绘图桌——它给了我们设计 Agent 如何思考、如何行动的工具。就像建筑师绘制蓝图，展示不同房间如何相连、人们如何在建筑中穿行一样，LangGraph 让我们设计不同能力如何相连、信息如何在 Agent 中流转。

### 核心特性：
- **状态管理（State Management）：** 跨交互维护持久化状态
- **灵活路由（Flexible Routing）：** 定义组件之间的复杂流程
- **持久化（Persistence）：** 保存和恢复工作流
- **可视化（Visualization）：** 查看并理解你的 Agent 结构

在本教程中，我们将构建一个多步骤的文本分析流水线来演示 LangGraph，该流水线通过三个阶段处理文本：
1. **文本分类（Text Classification）：** 将输入文本归类到预定义的类别中
2. **实体抽取（Entity Extraction）：** 从文本中识别关键实体
3. **文本摘要（Text Summarization）：** 生成输入文本的简洁摘要

这个流水线展示了如何用 LangGraph 为自然语言处理任务创建模块化、可扩展的工作流。


📖 **想了解更多关于构建 AI Agent 以及入门的背景知识，请查看我们详尽的博客文章：** [Your First AI Agent: Simpler Than You Think](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--langgraph-agent--langgraph-tutorial&click=substack-open&target=https%3A%2F%2Fopen.substack.com%2Fpub%2Fdiamantai%2Fp%2Fyour-first-ai-agent-simpler-than%3Fr%3D336pe4%26utm_campaign%3Dpost%26utm_medium%3Dweb%26showWelcomeOnShare%3Dfalse&text=Your%20First%20AI%20Agent%3A%20Simpler%20Than%20You%20Think)

*DiamantAI 是一份位居前 0.1% 的 AI 通讯，拥有超过 25,000 名订阅者，专注于 AI 技术、突破与教程。*

---


## 设置我们的环境

在深入代码之前，先来设置我们的开发环境。


In [ ]:
# Install required packages
!pip install langgraph langchain langchain-openai python-dotenv

### 设置 API Key

我们需要一个 OpenAI API key 才能使用它们的模型。如果你还没有，可以从 [https://platform.openai.com/signup](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--langgraph-agent--langgraph-tutorial&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fsignup&text=https%3A%2F%2Fplatform.openai.com%2Fsignup) 获取一个。

让我们加载环境变量：


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file (create this with your API key)
load_dotenv()

# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 测试我们的设置

让我们用 OpenAI 模型创建一个简单的测试，确认环境一切正常：


In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the ChatOpenAI instance
llm = ChatOpenAI(model="gpt-4o-mini")

# Test the setup
response = llm.invoke("Hello! Are you working?")
print(response.content)

## 构建我们的文本分析流水线

现在，为我们的 LangGraph 文本分析流水线导入所需的包：


In [ ]:
import os
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.runnables.graph import MermaidDrawMethod
from IPython.display import display, Image

### 设计 Agent 的记忆

正如人类的智能需要记忆一样，我们的 Agent 也需要一种方式来跟踪信息。我们用 TypedDict 定义状态结构来实现这一点：


In [ ]:
class State(TypedDict):
    text: str
    classification: str
    entities: List[str]
    summary: str

# Initialize our language model with temperature=0 for more deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 创建 Agent 的核心能力

现在我们来创建 Agent 实际使用的技能。每一项能力都实现为一个执行特定类型分析的函数。

#### 1. 分类节点


In [ ]:
def classification_node(state: State):
    '''Classify the text into one of the categories: News, Blog, Research, or Other'''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Classify the following text into one of the categories: News, Blog, Research, or Other.\n\nText:{text}\n\nCategory:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    classification = llm.invoke([message]).content.strip()
    return {"classification": classification}

#### 2. 实体抽取节点


In [ ]:
def entity_extraction_node(state: State):
    '''Extract all the entities (Person, Organization, Location) from the text'''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Extract all the entities (Person, Organization, Location) from the following text. Provide the result as a comma-separated list.\n\nText:{text}\n\nEntities:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    entities = llm.invoke([message]).content.strip().split(", ")
    return {"entities": entities}

#### 3. 摘要节点


In [ ]:
def summarization_node(state: State):
    '''Summarize the text in one short sentence'''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Summarize the following text in one short sentence.\n\nText:{text}\n\nSummary:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    summary = llm.invoke([message]).content.strip()
    return {"summary": summary}

### 把它们整合起来

接下来是最激动人心的部分——用 LangGraph 把这些能力连接成一个协同工作的系统：


In [ ]:
# Create our StateGraph
workflow = StateGraph(State)

# Add nodes to the graph
workflow.add_node("classification_node", classification_node)
workflow.add_node("entity_extraction", entity_extraction_node)
workflow.add_node("summarization", summarization_node)

# Add edges to the graph
workflow.set_entry_point("classification_node")  # Set the entry point of the graph
workflow.add_edge("classification_node", "entity_extraction")
workflow.add_edge("entity_extraction", "summarization")
workflow.add_edge("summarization", END)

# Compile the graph
app = workflow.compile()

### 可视化工作流

LangGraph 的强大特性之一就是能够可视化我们的工作流：


In [ ]:
# Display a visualization of our graph
try:
    display(
        Image(
            app.get_graph().draw_mermaid_png(
                draw_method=MermaidDrawMethod.API,
            )
        )
    )
except Exception as e:
    print(f"Error generating visualization: {e}")
    print("The graph structure is: classification_node -> entity_extraction -> summarization -> END")

## 测试我们的 Agent

既然我们已经构建好了 Agent，让我们用一个真实文本示例来看看它的表现：


In [ ]:
sample_text = """
OpenAI has announced the GPT-4 model, which is a large multimodal model that exhibits human-level performance on various professional benchmarks. It is developed to improve the alignment and safety of AI systems.
Additionally, the model is designed to be more efficient and scalable than its predecessor, GPT-3. The GPT-4 model is expected to be released in the coming months and will be available to the public for research and development purposes.
"""

state_input = {"text": sample_text}
result = app.invoke(state_input)

print("Classification:", result["classification"])
print("\nEntities:", result["entities"])
print("\nSummary:", result["summary"])

## 理解协同处理的威力

这个结果之所以令人印象深刻，并不仅仅在于各项单独的输出——而在于每一步如何在其他步骤的基础上构建，从而形成对文本的完整理解。

- **分类（classification）** 提供了帮助我们理解文本类型的上下文
- **实体抽取（entity extraction）** 识别出重要的名称和概念
- **摘要（summarization）** 提炼出文档的精髓

这与人类的阅读理解相呼应：我们会自然地形成对文本类型的理解，留意重要的名称和概念，并在脑海中形成摘要——同时保持这些不同理解层面之间的关联。


## 用你自己的文本试试

现在让我们用另一段你选择的文本样例来试试这个流水线：


In [ ]:
# Replace this with your own text to analyze
your_text = """
The recent advancements in quantum computing have opened new possibilities for cryptography and data security. 
Researchers at MIT and Google have demonstrated quantum algorithms that could potentially break current encryption methods.
However, they are also developing new quantum-resistant encryption techniques to protect data in the future.
"""

# Process the text through our pipeline
your_result = app.invoke({"text": your_text})

print("Classification:", your_result["classification"])
print("\nEntities:", your_result["entities"])
print("\nSummary:", your_result["summary"])

## 添加更多能力（进阶）

LangGraph 的强大之处之一在于，我们可以非常容易地为 Agent 扩展新的能力。让我们给流水线添加一个情感分析节点：


In [ ]:
# First, let's update our State to include sentiment
class EnhancedState(TypedDict):
    text: str
    classification: str
    entities: List[str]
    summary: str
    sentiment: str

# Create our sentiment analysis node
def sentiment_node(state: EnhancedState):
    '''Analyze the sentiment of the text: Positive, Negative, or Neutral'''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Analyze the sentiment of the following text. Is it Positive, Negative, or Neutral?\n\nText:{text}\n\nSentiment:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    sentiment = llm.invoke([message]).content.strip()
    return {"sentiment": sentiment}

# Create a new workflow with the enhanced state
enhanced_workflow = StateGraph(EnhancedState)

# Add the existing nodes
enhanced_workflow.add_node("classification_node", classification_node)
enhanced_workflow.add_node("entity_extraction", entity_extraction_node)
enhanced_workflow.add_node("summarization", summarization_node)

# Add our new sentiment node
enhanced_workflow.add_node("sentiment_analysis", sentiment_node)

# Create a more complex workflow with branches
enhanced_workflow.set_entry_point("classification_node")
enhanced_workflow.add_edge("classification_node", "entity_extraction")
enhanced_workflow.add_edge("entity_extraction", "summarization")
enhanced_workflow.add_edge("summarization", "sentiment_analysis")
enhanced_workflow.add_edge("sentiment_analysis", END)

# Compile the enhanced graph
enhanced_app = enhanced_workflow.compile()

### 测试增强后的 Agent


In [ ]:
# Try the enhanced pipeline with the same text
enhanced_result = enhanced_app.invoke({"text": sample_text})

print("Classification:", enhanced_result["classification"])
print("\nEntities:", enhanced_result["entities"])
print("\nSummary:", enhanced_result["summary"])
print("\nSentiment:", enhanced_result["sentiment"])

## 给 LangGraph 流水线添加条件边（进阶逻辑）


### 为什么需要**条件边**？

到目前为止，我们的图走的是一条固定的线性路径：

classification_node → entity_extraction → summarization → (sentiment)

但在实际应用中，我们常常希望只在需要时才运行某些步骤。例如：

- 只有当文本是新闻（News）或研究（Research）类文章时才抽取实体。

- 如果文本非常短，则跳过摘要。

- 为博客（Blog）类文章添加自定义处理。

LangGraph 通过条件边——基于当前状态数据动态路由执行的逻辑门——让这一切变得简单。


### 1. 更新状态
我们将使用进阶部分中的同一个 EnhancedState：


In [ ]:
class EnhancedState(TypedDict):
    text: str
    classification: str
    entities: List[str]
    summary: str
    sentiment: str

### 2. 创建路由函数
这个函数根据分类结果进行路由：


In [ ]:
# Route after classification
def route_after_classification(state: EnhancedState) -> str:
    category = state["classification"].lower() # returns: "news", "blog", "research", "other"
    return category in ["news", "research"]

### 3. 定义图


In [ ]:
from langgraph.graph import StateGraph, END

conditional_workflow = StateGraph(EnhancedState)

# Add nodes
conditional_workflow.add_node("classification_node", classification_node)
conditional_workflow.add_node("entity_extraction", entity_extraction_node)
conditional_workflow.add_node("summarization", summarization_node)
conditional_workflow.add_node("sentiment_analysis", sentiment_node)

# Set entry point
conditional_workflow.set_entry_point("classification_node")

# Add conditional edge
conditional_workflow.add_conditional_edges("classification_node", route_after_classification, path_map={
    True: "entity_extraction",
    False: "summarization"
})

# Add remaining static edges
conditional_workflow.add_edge("entity_extraction", "summarization")
conditional_workflow.add_edge("summarization", "sentiment_analysis")
conditional_workflow.add_edge("sentiment_analysis", END)

# Compile
conditional_app = conditional_workflow.compile()




### 4. 可视化


In [ ]:
from IPython.display import Image, display

try:
    display(Image(
        conditional_app.get_graph().draw_mermaid_png()
    ))
except:
    print("Graph: classification_node → (conditional) → [entity_extraction or summarization] → sentiment_analysis → END")


### 5. 测试条件流水线


In [ ]:
test_text = """
OpenAI released the GPT-4 model with enhanced performance on academic and professional tasks. It's seen as a major breakthrough in alignment and reasoning capabilities.
"""

result = conditional_app.invoke({"text": test_text})

print("Classification:", result["classification"])
print("Entities:", result.get("entities", "Skipped"))
print("Summary:", result["summary"])
print("Sentiment:", result["sentiment"])


现在用博客类文本试试：


In [ ]:
blog_text = """
Here's what I learned from a week of meditating in silence. No phones, no talking—just me, my breath, and some deep realizations.
"""

result = conditional_app.invoke({"text": blog_text})

print("Classification:", result["classification"])
print("Entities:", result.get("entities", "Skipped (not applicable)"))
print("Summary:", result["summary"])
print("Sentiment:", result["sentiment"])


有了条件边，我们的 Agent 现在可以：

- 根据上下文做决策

- 跳过不必要的步骤

- 运行得更快、更省

- 行为更加智能


## 结语

在本教程中，我们：

1. 探索了 LangGraph 的概念及其基于图的方式
2. 构建了一个包含分类、实体抽取和摘要的文本处理流水线
3. 为流水线增强了更多能力
4. 引入条件边，根据分类结果动态控制流程
5. 可视化我们的工作流
6. 用真实文本示例测试了我们的 Agent

LangGraph 通过把 AI Agent 建模为能力的图，为创建 Agent 提供了强大的框架。这种方式让设计、修改和扩展复杂的 AI 系统变得轻而易举。

### 下一步

- 添加更多节点来扩展 Agent 的能力
- 尝试不同的 LLM 和参数
- 探索 LangGraph 的状态持久化特性，用于持续的对话

更多信息，请查看 LangGraph 官方文档：https://langchain-ai.github.io/langgraph/
